# Michigan Traders — Module 3
# The 5 Pillars of a QuantConnect Algorithm  ·  *ANSWER KEY*

**Series:** MAT Education · QuantConnect Core
**Level:** Intermediate (builds on Modules 1–2; assumes Python fluency)
**Format:** **Reference + Fill-in-the-Pillar** — every pillar gets an explanation, worked examples, and a `# TODO` exercise for you.

---

Every QuantConnect algorithm — from a two-line moving-average crossover to a production-grade ML strategy — is assembled from the same **five building blocks**. Learn these five pillars and you can read, write, and extend *any* QC algo you encounter this year.

After working through each pillar individually you'll assemble the pieces into two complete, paste-ready algorithms: an MA crossover (Pillars 1–4) and a monthly cross-sectional momentum strategy (all 5 Pillars).

## How to use this notebook

> ⚠️ **Paste-into-IDE notebook.** The `QCAlgorithm` class runs inside the QuantConnect LEAN engine — cells that define algorithm classes will *not* execute in a plain local Jupyter kernel. Open [quantconnect.com](https://www.quantconnect.com), create a new algorithm project, and paste the code there to run a backtest.

For each pillar you'll see:

1. **An explanation** of what the pillar does and *why* it exists.
2. **Worked examples** — read the code and comments carefully; the same patterns repeat in every QC algo.
3. **✏️ Your turn** — an algorithm stub with `# TODO` blanks. Fill in the missing pillar.
4. **Verify** — compare your answer with `03_Five_Pillars_SOLUTIONS.ipynb` or paste it into QC and check it compiles.

Work top to bottom; use the cheat sheet at the end as a quick-reference lookup.

## The QuantConnect Lifecycle

One mental model to lock in before the five pillars:

```
  ┌──────────────────────────────────────────────────────────┐
  │                  Your Algorithm Class                    │
  │                                                          │
  │   initialize()         ← called ONCE at startup          │
  │       set dates, cash, assets, indicators                │
  │                                                          │
  │   on_data(data: Slice) ← called EVERY BAR               │
  │       read prices → check signals → place orders         │
  └──────────────────────────────────────────────────────────┘
```

`initialize` is your setup code — it runs once when the backtest begins. `on_data` is your main loop — QuantConnect calls it every time a new bar of data arrives (once per day for daily data, once per minute for minute data, and so on). Everything else — indicators, orders, portfolio checks — lives inside one of these two methods.

### The empty skeleton

Every QC algorithm starts with this two-method shell. The class inherits from `QCAlgorithm`, which provides all the tools you'll need — data feeds, order routing, portfolio tracking, logging, and more — as `self.*` methods.

In [ ]:
class MyAlgorithm(QCAlgorithm):
    # Minimal QC algorithm skeleton.
    # Copy this into any QuantConnect project as your starting point.

    def initialize(self):
        # Pillar 1: tell QC when to run, how much cash, which assets to load
        pass

    def on_data(self, data: Slice):
        # Pillar 2: handle new data
        # Pillar 3: check indicators
        # Pillar 4: place orders
        pass

## Pillar 1 — Initialize

`initialize` runs **once** at the very start of every backtest (or when you go live). It's where you configure everything QC needs before the first bar of data arrives:

- **Date range** — which period to backtest
- **Starting capital** — how much paper money to begin with
- **Asset subscriptions** — which tickers to load, at what data resolution
- **Indicators** — set up moving averages, RSI, etc. (Pillar 3 covers this in depth)
- **Benchmark** — what to compare performance against (default: SPY)

### Setting up dates and cash

Three methods, always called in this order:

| Method | What it does |
|---|---|
| `self.set_start_date(year, month, day)` | First bar of backtest data |
| `self.set_end_date(year, month, day)` | Last bar (omit → today) |
| `self.set_cash(amount)` | Starting portfolio value in USD |

In [ ]:
class Pillar1_DatesAndCash(QCAlgorithm):
    def initialize(self):
        # ── dates ──────────────────────────────────────────────────────────
        self.set_start_date(2022, 1, 1)   # backtest starts 2022-01-01
        self.set_end_date(2024, 12, 31)   # backtest ends   2024-12-31

        # ── capital ────────────────────────────────────────────────────────
        self.set_cash(100_000)            # $100,000 starting cash

        # No assets subscribed yet — this algo does nothing, but it's valid QC code.

### Subscribing to data with add_equity

`add_equity` tells QC which ticker to load price data for. It returns an `Equity` object; we call `.symbol` on it and store the result on `self` so we can reference the same symbol handle throughout the algorithm.

```
self.symbol = self.add_equity(ticker, resolution).symbol
```

| Argument | Example | Notes |
|---|---|---|
| `ticker` | `"SPY"` | US equities and ETFs only in this course |
| `resolution` | `Resolution.DAILY` | `DAILY` / `HOUR` / `MINUTE` (all UPPER_SNAKE_CASE enums) |

In [ ]:
class Pillar1_AddEquity(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)

        # add_equity returns an Equity object; .symbol gives us a Symbol handle.
        # We save it on self so on_data can use it without re-looking up the ticker.
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # Optional: benchmark against QQQ instead of the default SPY
        self.set_benchmark("QQQ")

### ✏️ Your turn — initialize a two-stock strategy

Set up an algorithm that:
- Runs from **2020-01-01** to **2023-12-31**
- Starts with **$50,000** cash
- Loads **AAPL** and **TLT** at daily resolution, storing their Symbol objects as `self.aapl` and `self.tlt`

Write only the `initialize` method body — the rest of the class is already provided.

In [ ]:
class TwoStockAlgo(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2020, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(50_000)
        self.aapl = self.add_equity("AAPL", Resolution.DAILY).symbol
        self.tlt  = self.add_equity("TLT",  Resolution.DAILY).symbol

    def on_data(self, data: Slice):   # Pillar 2 — provided for context
        pass

> **Verify:** Your `initialize` should call exactly four methods: `set_start_date`, `set_end_date`, `set_cash`, and `add_equity` (twice). Compare with `03_Five_Pillars_SOLUTIONS.ipynb`, or paste the class into a QC project — it should compile without errors.

## Pillar 2 — Data: on_data and the Slice

Every time a new bar of market data arrives, QC calls `on_data(self, data: Slice)`. The `Slice` is a snapshot of that bar — prices, volume, and any other data your algorithm subscribed to.

Think of `Slice` as a dictionary keyed by `Symbol`:
- `data[symbol]` → a `TradeBar` object with `.open`, `.close`, `.high`, `.low`, `.volume`
- `data.contains_key(symbol)` → check that data actually arrived (not every symbol trades every bar)

### The guard pattern

Always confirm that the data you need arrived before using it. Missing data is normal — exchanges are closed, trading halts occur, newly added universe members haven't loaded yet.

Two equivalent ways to guard:

```python
if self.symbol not in data:             # Python "in" operator — most readable
    return

if not data.contains_key(self.symbol):  # explicit QC method — same result
    return
```

Both short-circuit `on_data` for that bar gracefully.

### What's in a TradeBar

| Attribute | Type | What it holds |
|---|---|---|
| `bar.open` | `float` | opening price of the bar |
| `bar.high` | `float` | highest price of the bar |
| `bar.low` | `float` | lowest price of the bar |
| `bar.close` | `float` | closing price of the bar |
| `bar.volume` | `float` | total shares traded |
| `bar.time` | `datetime` | timestamp of the bar |

In [ ]:
class Pillar2_ReadPrice(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

    def on_data(self, data: Slice):
        # Guard: skip bars where SPY data didn't arrive
        if self.symbol not in data:
            return

        bar    = data[self.symbol]            # TradeBar for this bar
        close  = bar.close                    # closing price
        volume = bar.volume                   # shares traded that day

        # self.log() writes a line to the QC backtest log panel
        self.log(f"SPY | close={close:.2f}  volume={volume:,.0f}")

### ✏️ Your turn — log AAPL's daily closing price

Complete `on_data` so it:
- Guards against missing AAPL data
- Reads the closing price from the `Slice`
- Logs a message in exactly this format: `AAPL close: 182.45` (two decimal places)

The `initialize` method is already filled in for you.

In [ ]:
class Pillar2Exercise(QCAlgorithm):
    def initialize(self):                  # Pillar 1 — already done
        self.set_start_date(2023, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("AAPL", Resolution.DAILY).symbol

    def on_data(self, data: Slice):
        if self.symbol not in data:
            return
        close = data[self.symbol].close
        self.log(f"AAPL close: {close:.2f}")

> **Verify:** Your `on_data` should have three parts: a guard `return`, a `.close` read, and a `self.log(f"...")` call. Compare with the solutions file, then paste into QC — the backtest log should fill with daily AAPL prices.

## Pillar 3 — Indicators

Indicators transform raw price data into trading signals — a moving average smooths price noise, RSI measures momentum, Bollinger Bands quantify volatility. QuantConnect ships 100+ built-in indicators.

**Two ways to create an indicator:**

1. **Helper method (recommended):** `self.sma(symbol, period, resolution)` — QC automatically wires up the data feed.
2. **Manual:** `SimpleMovingAverage(period)` + `self.register_indicator(symbol, indicator, resolution)` — more control, same end result.

We use helper methods throughout this course.

### Creating indicators in initialize

Indicators are created in `initialize` (not in `on_data`) and stored on `self` so every call to `on_data` can access them. Common helpers:

| Helper | What it creates |
|---|---|
| `self.sma(symbol, period, resolution)` | Simple Moving Average |
| `self.ema(symbol, period, resolution)` | Exponential Moving Average |
| `self.rsi(symbol, period, resolution)` | Relative Strength Index (0–100) |
| `self.bb(symbol, period, k, resolution)` | Bollinger Bands |
| `self.macd(symbol, fast, slow, signal, resolution)` | MACD |

In [ ]:
class Pillar3_CreateIndicators(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # Create a 20-day and a 50-day simple moving average.
        # QC automatically feeds SPY's daily closing prices into both indicators.
        self.sma_20 = self.sma(self.symbol, 20, Resolution.DAILY)
        self.sma_50 = self.sma(self.symbol, 50, Resolution.DAILY)

        # Warm up: pre-load 50 bars of history so both indicators are ready
        # on the very first on_data call.
        self.set_warm_up(50)

### Reading indicator values in on_data

An indicator has two key attributes:

| Attribute | Type | Meaning |
|---|---|---|
| `.is_ready` | `bool` | `True` once enough bars have been processed (e.g. 20 bars for SMA20) |
| `.current.value` | `float` | The most recently computed value |

**Always check `.is_ready` before using `.current.value`.** An unready indicator returns 0, which would produce garbage signals on the first few bars.

In [ ]:
class Pillar3_ReadIndicators(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.sma_20 = self.sma(self.symbol, 20, Resolution.DAILY)
        self.sma_50 = self.sma(self.symbol, 50, Resolution.DAILY)
        self.set_warm_up(50)

    def on_data(self, data: Slice):
        if self.symbol not in data:
            return

        # Guard: wait until both indicators have processed enough history
        if not self.sma_20.is_ready or not self.sma_50.is_ready:
            return

        fast  = self.sma_20.current.value   # today's 20-day SMA value
        slow  = self.sma_50.current.value   # today's 50-day SMA value
        price = data[self.symbol].close

        self.log(f"SPY={price:.2f}  SMA20={fast:.2f}  SMA50={slow:.2f}  "
                 f"signal={'BULL' if fast > slow else 'BEAR'}")

In [ ]:
# Why set_warm_up matters
#
# set_warm_up(n) pre-loads n bars of historical data before the backtest
# date range begins, so indicators are already computed on day 1.
#
# Without warm_up: on_data fires from day 1 but .is_ready stays False for
# the first (period - 1) bars -- you'd silently skip early signals.
#
# Rule of thumb: set_warm_up to the longest indicator period you use.
#   Two indicators with periods 20 and 50 -> set_warm_up(50).

### ✏️ Your turn — add an RSI indicator

Complete the algorithm below. In `initialize`, add an **RSI with period 14**:
```python
self.rsi_14 = self.rsi(self.symbol, 14, Resolution.DAILY)
```
Then in `on_data`:
- Guard against missing data AND an unready RSI
- Read the RSI value and log: `RSI=67.3` (one decimal place)

In [ ]:
class Pillar3Exercise(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.rsi_14 = self.rsi(self.symbol, 14, Resolution.DAILY)
        self.set_warm_up(14)

    def on_data(self, data: Slice):
        if self.symbol not in data:
            return
        if not self.rsi_14.is_ready:
            return
        rsi_val = self.rsi_14.current.value
        self.log(f"RSI={rsi_val:.1f}")

> **Verify:** Your `initialize` should call `self.rsi(...)` and `self.set_warm_up(14)`. Your `on_data` should have two guards and one `self.log(...)`. Compare with the solutions file.

## Pillar 4 — Portfolio & Orders

With a signal in hand (Pillar 3), you need to act on it. Pillar 4 covers:
- **Checking positions** — are we already in this trade?
- **Placing orders** — enter and exit positions
- **Reading portfolio state** — total value, cash on hand

### Checking portfolio state

| Expression | Returns | What it means |
|---|---|---|
| `self.portfolio[symbol].invested` | `bool` | `True` if we hold any shares |
| `self.portfolio[symbol].quantity` | `float` | Shares held (negative = short) |
| `self.portfolio.total_portfolio_value` | `float` | Cash + all open positions (USD) |
| `self.portfolio.cash` | `float` | Uninvested cash |

In [ ]:
class Pillar4_CheckPortfolio(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

    def on_data(self, data: Slice):
        if self.symbol not in data:
            return

        invested = self.portfolio[self.symbol].invested    # bool
        qty      = self.portfolio[self.symbol].quantity    # shares
        total    = self.portfolio.total_portfolio_value    # USD
        cash     = self.portfolio.cash                     # USD

        self.log(f"invested={invested}  qty={qty}  total=${total:,.0f}  cash=${cash:,.0f}")

### Placing orders

Three methods cover 90% of use cases:

| Method | What it does |
|---|---|
| `self.set_holdings(symbol, weight)` | Target `weight` fraction of portfolio in `symbol` (0 = flat, 1.0 = 100% long, -1 = 100% short) |
| `self.liquidate(symbol)` | Close the entire position in `symbol` immediately |
| `self.market_order(symbol, quantity)` | Buy/sell exactly `quantity` shares at market price |

**`set_holdings` is the most common** for single-asset and multi-asset strategies because it handles position sizing automatically based on current portfolio value.

In [ ]:
class Pillar4_PlaceOrders(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol  = self.add_equity("SPY", Resolution.DAILY).symbol
        self.sma_20  = self.sma(self.symbol, 20, Resolution.DAILY)
        self.set_warm_up(20)

    def on_data(self, data: Slice):
        if self.symbol not in data or not self.sma_20.is_ready:
            return

        price    = data[self.symbol].close
        sma      = self.sma_20.current.value
        invested = self.portfolio[self.symbol].invested

        # Buy signal: price above 20-day average -> go 100% long
        if price > sma and not invested:
            self.set_holdings(self.symbol, 1.0)
            self.log(f"BUY  SPY @ {price:.2f}  (SMA20={sma:.2f})")

        # Sell signal: price fell below the average -> exit
        elif price < sma and invested:
            self.liquidate(self.symbol)
            self.log(f"SELL SPY @ {price:.2f}  (SMA20={sma:.2f})")

### ✏️ Your turn — RSI mean-reversion trade logic

Complete `on_data` below so it:
- **Buys** SPY (100% allocation) when RSI < 30 AND not already long
- **Sells** (liquidates) when RSI > 70 AND currently long
- Logs each trade in the format: `BUY SPY RSI=28.4` or `SELL SPY RSI=71.2`

In [ ]:
class Pillar4Exercise(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol  = self.add_equity("SPY", Resolution.DAILY).symbol
        self.rsi_14  = self.rsi(self.symbol, 14, Resolution.DAILY)
        self.set_warm_up(14)

    def on_data(self, data: Slice):
        if self.symbol not in data or not self.rsi_14.is_ready:
            return

        rsi_val  = self.rsi_14.current.value
        invested = self.portfolio[self.symbol].invested

        if rsi_val < 30 and not invested:
            self.set_holdings(self.symbol, 1.0)
            self.log(f"BUY SPY RSI={rsi_val:.1f}")
        elif rsi_val > 70 and invested:
            self.liquidate(self.symbol)
            self.log(f"SELL SPY RSI={rsi_val:.1f}")

> **Verify:** Your `on_data` should have two `if/elif` branches — one calling `set_holdings`, one calling `liquidate`. Compare with the solutions file, then paste into QC and run a backtest to see trades in the log.

## Pillar 5 — Universe Selection & Scheduling

Pillars 1–4 are sufficient for single-stock strategies. Pillar 5 adds two power tools:

1. **Universe Selection** — instead of hardcoding tickers, let QC dynamically choose which stocks to trade (e.g. top 50 by dollar volume, updated each day).
2. **Scheduling** — trigger a function at a specific recurring time (e.g. rebalance on the first trading day of every month, 30 minutes after the open).

Together, these are what separate a toy algo from a production cross-sectional strategy.

### Universe Selection

`add_universe(filter_fn)` subscribes to a *dynamic* set of securities. QC calls `filter_fn` each day with a list of all tradeable US stocks (`CoarseFundamental` objects) and expects back a list of `Symbol` objects to include.

```python
def coarse_filter(self, coarse):
    # Sort by dollar volume descending, return top 20 symbols
    sorted_stocks = sorted(coarse, key=lambda x: x.dollar_volume, reverse=True)
    return [x.symbol for x in sorted_stocks[:20]]
```

When stocks enter or leave the universe, QC calls `on_securities_changed`:

```python
def on_securities_changed(self, changes):
    for security in changes.added_securities:
        self.log(f"Added: {security.symbol}")
    for security in changes.removed_securities:
        self.liquidate(security.symbol)   # exit position when stock leaves
```

### Scheduling

`self.schedule.on(date_rule, time_rule, function)` calls a function at a recurring time. Common rules:

| Date rule | Fires on... |
|---|---|
| `self.date_rules.every_day()` | Every trading day |
| `self.date_rules.week_start()` | First trading day of each week |
| `self.date_rules.month_start()` | First trading day of each month |

| Time rule | Fires at... |
|---|---|
| `self.time_rules.after_market_open(symbol, n)` | n minutes after open |
| `self.time_rules.before_market_close(symbol, n)` | n minutes before close |
| `self.time_rules.at(hour, minute)` | Specific time (Eastern) |

In [ ]:
class Pillar5_UniverseAndSchedule(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2021, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)

        # SPY used only as a scheduling reference (market-hours anchor)
        self.spy = self.add_equity("SPY", Resolution.DAILY).symbol

        # Universe: top 30 US stocks by dollar volume, refiltered each day by QC
        self.universe_symbols = []
        self.add_universe(self.coarse_filter)

        # Schedule: call self.rebalance on the first trading day of each month,
        # 30 minutes after SPY opens
        self.schedule.on(
            self.date_rules.month_start(),
            self.time_rules.after_market_open(self.spy, 30),
            self.rebalance
        )

    def coarse_filter(self, coarse):
        # Keep only stocks with fundamental data and a price above $5
        eligible = [x for x in coarse if x.has_fundamental_data and x.price > 5]
        return [x.symbol for x in sorted(eligible,
                                         key=lambda x: x.dollar_volume,
                                         reverse=True)[:30]]

    def on_securities_changed(self, changes):
        for security in changes.added_securities:
            if security.symbol not in self.universe_symbols:
                self.universe_symbols.append(security.symbol)
        for security in changes.removed_securities:
            self.liquidate(security.symbol)
            if security.symbol in self.universe_symbols:
                self.universe_symbols.remove(security.symbol)

    def rebalance(self):
        if not self.universe_symbols:
            return
        weight = 1.0 / len(self.universe_symbols)
        for symbol in self.universe_symbols:
            self.set_holdings(symbol, weight)
        self.log(f"Rebalanced: {len(self.universe_symbols)} stocks at {weight:.1%} each")

    def on_data(self, data: Slice):
        pass   # all trading happens in rebalance(), not on_data

### ✏️ Your turn — add a weekly rebalance schedule

The algorithm below has a universe and a `rebalance` function but no scheduler.
Add a `self.schedule.on(...)` call in `initialize` (after `self.add_universe(...)`) so it triggers `self.rebalance`:
- **Date rule:** `self.date_rules.week_start()` — first trading day of each week
- **Time rule:** `self.time_rules.after_market_open(self.spy, 0)` — at market open

The three-line `self.schedule.on(...)` block is all that's needed.

In [ ]:
class Pillar5Exercise(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2021, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)
        self.spy = self.add_equity("SPY", Resolution.DAILY).symbol
        self.add_universe(self.coarse_filter)
        self.universe_symbols = []
        self.schedule.on(
            self.date_rules.week_start(),
            self.time_rules.after_market_open(self.spy, 0),
            self.rebalance
        )

    def coarse_filter(self, coarse):
        eligible = [x for x in coarse if x.has_fundamental_data and x.price > 10]
        return [x.symbol for x in sorted(eligible,
                                         key=lambda x: x.dollar_volume,
                                         reverse=True)[:20]]

    def on_securities_changed(self, changes):
        for s in changes.added_securities:
            if s.symbol not in self.universe_symbols:
                self.universe_symbols.append(s.symbol)
        for s in changes.removed_securities:
            self.liquidate(s.symbol)
            if s.symbol in self.universe_symbols:
                self.universe_symbols.remove(s.symbol)

    def rebalance(self):
        if not self.universe_symbols:
            return
        weight = 1.0 / len(self.universe_symbols)
        for sym in self.universe_symbols:
            self.set_holdings(sym, weight)
        self.log(f"Rebalanced {len(self.universe_symbols)} stocks")

    def on_data(self, data: Slice):
        pass

> **Verify:** Your `initialize` should contain a `self.schedule.on(...)` call with `date_rules.week_start()`, `time_rules.after_market_open(self.spy, 0)`, and `self.rebalance`. Compare with the solutions file.

## Capstone 1 — Moving Average Crossover (Pillars 1–4)

The classic algo-trading starter: go long when a fast SMA crosses above a slow SMA ("golden cross"), exit when it crosses back below ("death cross"). Each pillar is labelled in the comments so you can trace how the five building blocks fit together.

**How to run it:** copy the entire class into a new QuantConnect algorithm project and click **Backtest**.

Read through the code top to bottom before running. Notice how the pillar labels map directly to the five sections you just studied.

In [ ]:
class MovingAverageCrossover(QCAlgorithm):
    # MA crossover on SPY using Pillars 1 through 4.

    # ── Pillar 1: Initialize ─────────────────────────────────────────────────
    def initialize(self):
        self.set_start_date(2020, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # ── Pillar 3: Indicators ─────────────────────────────────────────────
        self.fast = self.sma(self.symbol, 20, Resolution.DAILY)   # 20-day SMA
        self.slow = self.sma(self.symbol, 50, Resolution.DAILY)   # 50-day SMA
        self.set_warm_up(50)   # pre-load 50 bars so both SMAs are ready on day 1

    # ── Pillar 2: Data ───────────────────────────────────────────────────────
    def on_data(self, data: Slice):
        if self.symbol not in data:
            return

        # ── Pillar 3: Check indicators ────────────────────────────────────────
        if not self.fast.is_ready or not self.slow.is_ready:
            return

        fast_val = self.fast.current.value
        slow_val = self.slow.current.value
        price    = data[self.symbol].close

        # ── Pillar 4: Portfolio & Orders ──────────────────────────────────────
        invested = self.portfolio[self.symbol].invested

        # Golden cross: fast SMA moves above slow SMA -> enter long
        if fast_val > slow_val and not invested:
            self.set_holdings(self.symbol, 1.0)
            self.log(f"BUY  SPY @ {price:.2f}  (SMA20={fast_val:.2f} > SMA50={slow_val:.2f})")

        # Death cross: fast SMA drops below slow SMA -> exit
        elif fast_val < slow_val and invested:
            self.liquidate(self.symbol)
            self.log(f"SELL SPY @ {price:.2f}  (SMA20={fast_val:.2f} < SMA50={slow_val:.2f})")

## Capstone 2 — Monthly Cross-Sectional Momentum (All 5 Pillars)

Cross-sectional momentum ranks stocks by recent return and bets the top performers continue to lead. This version:
- Builds a **dynamic universe** of liquid US stocks (Pillar 5 — Universe)
- Scores each stock by **12-minus-1-month return** (12-month total minus the most recent month, which reduces short-term reversal noise)
- **Rebalances monthly**, equal-weighting the top 20 (Pillar 5 — Scheduling)
- Uses Pillars 1–4 for setup, data history, and order execution

**How to run it:** paste the class into a QuantConnect project. Expect the backtest to take longer than Capstone 1 — it computes returns across ~200 stocks each month.

Read `initialize` first to understand the full setup, then trace how `rebalance` calls `self.history()` to compute the momentum signal. Notice that `on_data` is empty — all trading happens inside `rebalance`.

In [ ]:
class MonthlyMomentum(QCAlgorithm):
    # Cross-sectional momentum: top-20 US stocks by 12-1m return, rebalanced monthly.

    # ── Pillar 1: Initialize ─────────────────────────────────────────────────
    def initialize(self):
        self.set_start_date(2019, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)

        # SPY as a market-hours anchor for scheduling; may appear in universe too
        self.spy = self.add_equity("SPY", Resolution.DAILY).symbol

        # ── Pillar 5: Universe ────────────────────────────────────────────────
        self.universe_symbols = []
        self.add_universe(self.coarse_filter)

        # ── Pillar 5: Schedule ────────────────────────────────────────────────
        self.schedule.on(
            self.date_rules.month_start(),
            self.time_rules.after_market_open(self.spy, 30),
            self.rebalance
        )

    # ── Pillar 5: Universe filter ────────────────────────────────────────────
    def coarse_filter(self, coarse):
        # Liquid stocks: has fundamental data, price > $5, dollar volume > $1M/day
        eligible = [x for x in coarse
                    if x.has_fundamental_data
                    and x.price > 5
                    and x.dollar_volume > 1_000_000]
        # Pool of top 200 by dollar volume -- large enough to rank for momentum
        return [x.symbol for x in sorted(eligible,
                                         key=lambda x: x.dollar_volume,
                                         reverse=True)[:200]]

    # ── Pillar 5: Universe events ─────────────────────────────────────────────
    def on_securities_changed(self, changes):
        for security in changes.added_securities:
            if security.symbol not in self.universe_symbols:
                self.universe_symbols.append(security.symbol)
        for security in changes.removed_securities:
            self.liquidate(security.symbol)
            if security.symbol in self.universe_symbols:
                self.universe_symbols.remove(security.symbol)

    # ── Pillar 2 + 4: Scheduled rebalance ────────────────────────────────────
    def rebalance(self):
        if len(self.universe_symbols) < 20:
            return   # not enough stocks to rank yet

        # history() returns a pandas DataFrame with a (symbol, time) multi-index
        history = self.history(self.universe_symbols, 252, Resolution.DAILY)

        scores = {}
        for symbol in self.universe_symbols:
            try:
                closes = history.loc[symbol]["close"]
                if len(closes) < 22:
                    continue
                ret_12m = closes.iloc[-1] / closes.iloc[0]   - 1   # 12-month return
                ret_1m  = closes.iloc[-1] / closes.iloc[-22] - 1   # last-month return
                scores[symbol] = ret_12m - ret_1m                   # 12-1 momentum score
            except Exception:
                continue

        if not scores:
            return

        # ── Pillar 4: Orders ──────────────────────────────────────────────────
        top20  = sorted(scores, key=scores.get, reverse=True)[:20]
        weight = 1.0 / len(top20)

        # Exit positions no longer in the top-20
        for symbol in self.universe_symbols:
            if symbol not in top20 and self.portfolio[symbol].invested:
                self.liquidate(symbol)

        # Enter or reweight top-20 at equal weight
        for symbol in top20:
            self.set_holdings(symbol, weight)

        self.log(f"Rebalanced: top {len(top20)} momentum stocks at {weight:.1%} each")

    def on_data(self, data: Slice):
        pass   # all trading happens in rebalance()

## Cheat sheet

| Pillar | Method / Pattern | What it does |
|---|---|---|
| **1 — Initialize** | `self.set_start_date(y, m, d)` | Backtest start date |
| | `self.set_end_date(y, m, d)` | Backtest end date |
| | `self.set_cash(n)` | Starting capital in USD |
| | `self.add_equity("SPY", Resolution.DAILY).symbol` | Subscribe to daily data |
| | `self.set_warm_up(n)` | Pre-load n bars of history |
| | `self.set_benchmark("QQQ")` | Custom benchmark |
| **2 — Data** | `def on_data(self, data: Slice):` | Called every bar |
| | `if self.symbol not in data: return` | Guard against missing data |
| | `data[symbol].close / .open / .high / .low / .volume` | Price fields |
| | `self.log(message)` | Write to backtest log |
| **3 — Indicators** | `self.sma(symbol, period, Resolution.DAILY)` | Simple Moving Average |
| | `self.ema(symbol, period)` | Exponential Moving Average |
| | `self.rsi(symbol, period)` | RSI (0–100) |
| | `self.bb(symbol, period, k)` | Bollinger Bands |
| | `.is_ready` / `.current.value` | Check readiness / read value |
| **4 — Portfolio** | `self.portfolio[symbol].invested` | Currently holding? |
| | `self.set_holdings(symbol, weight)` | Target weight (0 = flat, 1.0 = fully long) |
| | `self.liquidate(symbol)` | Exit position |
| | `self.market_order(symbol, qty)` | Buy/sell exact shares |
| **5 — Universe** | `self.add_universe(filter_fn)` | Dynamic stock selection |
| | `def on_securities_changed(self, changes):` | Handle entries/exits |
| | `self.schedule.on(date_rule, time_rule, fn)` | Recurring function |
| | `self.date_rules.month_start()` | First day of each month |
| | `self.time_rules.after_market_open(symbol, n)` | n minutes after open |

## Stretch goals

Bring these to the next meeting:

1. **Vary Capstone 1 parameters** — change the fast/slow periods (try 10/30 or 5/20). Does the crossover become more or less profitable? More or fewer trades per year?
2. **Add a volatility filter to Capstone 1** — only enter when ATR is below a threshold. Use `self.atr(self.symbol, 14)` and check `.current.value` against a fixed level.
3. **Modify Capstone 2's lookback** — try 6-month momentum (126 bars) instead of 12-month (252 bars). Does shorter-term momentum perform better or worse?
4. **Add a stop-loss to Capstone 1** — if the position falls more than 5% from entry, liquidate. Track the entry price on `self` and compare it to the current close in `on_data`.
5. **Combine Pillars 4 and 5** — use Capstone 2's universe but add an RSI filter in `rebalance`: only include top-20 stocks where RSI < 60 (avoid overbought entries).

## What's next

**Module 4 — The Research Environment & Working with Financial Data** picks up where Module 2's pandas skills left off and brings them into QuantConnect's interactive Research Environment. You'll use `QuantBook` to pull real market data, compute return series, run rolling statistics, and plot correlation matrices — all live in a Jupyter-style notebook on QuantConnect's servers.

**Official docs:**
- [Initialization](https://www.quantconnect.com/docs/v2/writing-algorithms/initialization)
- [Handling Data / Slice](https://www.quantconnect.com/docs/v2/writing-algorithms/securities/handling-data)
- [Indicators](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/supported-indicators)
- [Portfolio & Orders](https://www.quantconnect.com/docs/v2/writing-algorithms/trading-and-orders/order-types/market-orders)
- [Universe Selection](https://www.quantconnect.com/docs/v2/writing-algorithms/universes/equity/coarse-universe-selection)
- [Scheduling](https://www.quantconnect.com/docs/v2/writing-algorithms/scheduled-events)
- [PEP8 snake_case migration](https://www.quantconnect.com/announcements/16830/pep8-python-api-migration/)

*MAT Education · QuantConnect Core · Module 3.*